# CV Improver - Multi-LLM Comparison

Paste your CV and get improvement suggestions from 3 different LLMs side by side.
Uses OpenRouter to access multiple models through a single API.

**What this project demonstrates:**
- Calling multiple LLM providers (OpenAI, Claude, Gemini) via OpenRouter
- Comparing outputs from different models on the same task
- Structured prompt engineering for consistent, actionable feedback
- Practical benchmarking of LLM quality for a real-world use case

Built as part of Module 4 (Benchmarking LLMs) of the LLM & Agentic AI Bootcamp.

## Setup

OpenRouter provides a unified API compatible with OpenAI's client.
We just change the `base_url` and use our OpenRouter key - then we can call any model.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv()

# OpenRouter uses the same OpenAI client - just a different base_url
router_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

print("OpenRouter client ready!")

OpenRouter client ready!


## Models Configuration

We'll compare 3 top models from different providers, all through the same API.

In [2]:
MODELS = {
    "GPT-4o": "openai/gpt-4o",
    "Claude Sonnet": "anthropic/claude-sonnet-4",
    "Gemini 2.5 Pro": "google/gemini-2.5-pro-preview"
}

print("Models to compare:")
for name, model_id in MODELS.items():
    print(f"  - {name} ({model_id})")

Models to compare:
  - GPT-4o (openai/gpt-4o)
  - Claude Sonnet (anthropic/claude-sonnet-4)
  - Gemini 2.5 Pro (google/gemini-2.5-pro-preview)


## The CV Review Prompt

A structured prompt ensures each model returns feedback in the same format,
making comparison fair and easy.

In [3]:
REVIEW_PROMPT = """
You are an expert career coach and CV reviewer with 20 years of experience
in tech recruiting and HR.

Analyze the CV below and provide actionable feedback in this exact format:

## Overall Score: X/10

## Strengths (top 3)
1. ...
2. ...
3. ...

## Areas to Improve (top 3)
1. ...
2. ...
3. ...

## Specific Suggestions
- For each section of the CV (summary, experience, skills, education), give one concrete suggestion
- Include a rewritten example where helpful

## Missing Elements
- List anything important that's missing from the CV

## ATS Compatibility
- Rate how well this CV would pass Applicant Tracking Systems (High/Medium/Low)
- Suggest keyword improvements if needed

Be direct, specific, and constructive. No generic advice.
"""

print("Review prompt ready!")

Review prompt ready!


## Review Function

This function sends the CV to a specific model via OpenRouter and returns the feedback.

In [4]:
def review_cv(cv_text, model_name, model_id):
    """Send CV to a model via OpenRouter and return the review."""
    print(f"Asking {model_name}...")
    
    try:
        response = router_client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": REVIEW_PROMPT},
                {"role": "user", "content": f"Here is my CV:\n\n{cv_text}"}
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    
    except Exception as e:
        return f"Error with {model_name}: {e}"

In [5]:
def compare_reviews(cv_text):
    """Get reviews from all 3 models and display them side by side."""
    reviews = {}
    
    for name, model_id in MODELS.items():
        reviews[name] = review_cv(cv_text, name, model_id)
    
    # Display each review
    for name, review in reviews.items():
        display(Markdown(f"---\n# {name}\n{review}"))
    
    return reviews

## Try It!

Paste your CV text below (or use the sample) and run the cell.
All 3 models will review it and you can compare their feedback.

In [6]:
# Paste your CV here, or use this sample
my_cv = """
MARIO ROSSI
Email: mario.rossi@email.com | Phone: +39 333 1234567 | Milan, Italy

SUMMARY
Software developer with 3 years of experience. Good at Python and JavaScript.
Looking for new opportunities.

EXPERIENCE
Junior Developer - TechCorp Srl, Milan
Jan 2022 - Present
- Wrote code for web applications
- Fixed bugs
- Worked with the team on projects
- Used Python and React

Intern - StartupXYZ, Rome
Jun 2021 - Dec 2021
- Helped with development tasks
- Learned about APIs
- Did some testing

EDUCATION
Bachelor in Computer Science - University of Milan, 2021

SKILLS
Python, JavaScript, React, SQL, Git, HTML/CSS

LANGUAGES
Italian (native), English (B2)
"""

print(f"CV length: {len(my_cv.split())} words")
print("Starting multi-model review...\n")
reviews = compare_reviews(my_cv)

CV length: 105 words
Starting multi-model review...

Asking GPT-4o...
Asking Claude Sonnet...
Asking Gemini 2.5 Pro...


---
# GPT-4o
## Overall Score: 5/10

## Strengths (top 3)
1. **Technical Skills**: The CV lists a solid foundation in programming languages and technologies, such as Python, JavaScript, and React.
2. **Relevant Experience**: Experience in both a junior developer role and an internship provides a practical background in the field.
3. **Clear Education Path**: A Bachelor's degree in Computer Science from a reputable university is well-aligned with the career path.

## Areas to Improve (top 3)
1. **Summary Section**: The summary is too vague and lacks details on specific achievements or a clear career objective.
2. **Experience Details**: The descriptions of roles and responsibilities are too generic and do not highlight specific contributions or achievements.
3. **Lack of Quantifiable Achievements**: There are no metrics or outcomes mentioned that demonstrate the impact of your work.

## Specific Suggestions

- **Summary**: Add more detail about your specific skills and career goals. Highlight a key achievement or project.
  - *Rewritten Example*: "Enthusiastic software developer with 3 years of experience specializing in Python and JavaScript. Successfully developed and deployed web applications that improved user engagement by 20%. Seeking to leverage expertise in a challenging new role to drive innovative solutions."

- **Experience**: Elaborate on your contributions and achievements. Use metrics where possible.
  - *Rewritten Example*: "Junior Developer - TechCorp Srl, Milan (Jan 2022 - Present): Developed and maintained web applications using Python and React, resulting in a 15% reduction in load time. Collaborated with a team of 5 to launch a new feature that increased user retention by 10%."

- **Skills**: Consider organizing skills into categories (e.g., Programming Languages, Frameworks, Tools) for clarity.
  - *Example*: 
    - Programming Languages: Python, JavaScript
    - Frameworks: React
    - Tools: SQL, Git, HTML/CSS

- **Education**: Mention any relevant coursework, projects, or honors that could strengthen your profile.
  - *Example*: "Bachelor in Computer Science - University of Milan, 2021: Relevant coursework included Data Structures, Algorithms, and Web Development. Developed a capstone project that involved creating a fully functional e-commerce website."

## Missing Elements
- **Projects**: Include a section for personal or academic projects that showcase your skills.
- **Certifications**: If applicable, list any relevant certifications that could enhance your profile.
- **Professional Development**: Mention any workshops, seminars, or online courses that demonstrate your commitment to continuous learning.

## ATS Compatibility
- **Rating**: Medium
- **Suggestions**: Use more industry-specific keywords and phrases to improve ATS compatibility, such as "full-stack development," "API integration," "agile methodologies," and "version control."

By implementing these changes, your CV will better reflect your skills and achievements, making it more compelling to potential employers.

---
# Claude Sonnet
## Overall Score: 4/10

## Strengths (top 3)
1. **Clean, readable format** - The CV has a clear structure that's easy to scan
2. **Relevant technical skills** - Shows knowledge of in-demand technologies (Python, React, JavaScript)
3. **Recent experience** - Current employment demonstrates active engagement in the field

## Areas to Improve (top 3)
1. **Extremely vague descriptions** - No quantifiable achievements or specific impact mentioned
2. **Weak professional summary** - Generic and doesn't differentiate you from other candidates
3. **Missing critical details** - No projects, certifications, or concrete examples of work

## Specific Suggestions

**Summary**: Rewrite to be specific and value-focused:
*"Full-stack developer with 3+ years building scalable web applications using Python/Django and React. Delivered 15+ features serving 10K+ users at TechCorp, with focus on API development and database optimization. Seeking senior developer role to leverage expertise in modern web technologies."*

**Experience**: Add metrics and specific technologies:
- Instead of "Wrote code for web applications" → "Developed 8 REST APIs using Django/PostgreSQL, reducing data retrieval time by 40%"
- Add specific projects, team size, technologies used

**Skills**: Organize by category and add proficiency levels:
- **Backend**: Python (Django/Flask), SQL (PostgreSQL, MySQL)
- **Frontend**: JavaScript (ES6+), React, HTML5/CSS3
- **Tools**: Git, Docker, AWS (if applicable)

**Education**: Add GPA if strong (>3.5), relevant coursework, or thesis topic

## Missing Elements
- **Projects section** - Personal or professional projects with GitHub links
- **Certifications** - Any relevant tech certifications
- **Achievements/metrics** - Performance improvements, user impact, efficiency gains
- **LinkedIn profile** - Essential for tech professionals
- **Portfolio/GitHub link** - Critical for developers

## ATS Compatibility
**Rating: Low**

**Keyword improvements needed:**
- Add framework names (Django, Flask, Node.js)
- Include methodology keywords (Agile, Scrum, CI/CD)
- Add database technologies (PostgreSQL, MongoDB)
- Include cloud platforms if used (AWS, Azure, GCP)
- Add testing frameworks (Jest, pytest, unittest)

The CV needs significant expansion with concrete examples and technical depth to compete effectively.

---
# Gemini 2.5 Pro
Of course. Based on my 20 years of experience in tech recruiting, here is a direct and actionable analysis of Mario's CV.

***

## Overall Score: 4/10

This CV functions as a basic record of employment but fails to market you as a valuable candidate. It tells a recruiter *what* you did, but not *how well* you did it or what *impact* you had. It would likely be overlooked in a competitive applicant pool.

## Strengths (top 3)
1.  **Clear & Simple Layout:** The CV is easy to read and uses standard section headings. A recruiter can find information quickly.
2.  **Relevant Tech Stack:** The skills listed (Python, JavaScript, React, SQL) are in high demand for junior to mid-level developer roles.
3.  **Correct Chronology:** The experience is listed in reverse chronological order, which is the industry standard and expected by recruiters.

## Areas to Improve (top 3)
1.  **Lack of Quantifiable Impact:** The experience section is a list of tasks, not achievements. Phrases like "Wrote code" and "Fixed bugs" are meaningless without context. You need to show the *result* of your work using metrics (e.g., performance improvements, features launched, bugs reduced).
2.  **Passive and Vague Language:** The descriptions use weak phrases like "Helped with" and "Worked with." You need to use strong, specific action verbs that demonstrate ownership and capability (e.g., "Engineered," "Developed," "Resolved," "Collaborated").
3.  **Ineffective Summary:** The summary is generic and adds no value. "Looking for new opportunities" is obvious and wastes valuable space. This section should be a powerful 2-3 line pitch that highlights your key skills and career goals.

## Specific Suggestions

### Summary
Your summary is the first thing a recruiter reads. It must be a compelling pitch. Replace the generic statement with a targeted summary that highlights your core competencies and what you bring to the table.

**Rewritten Example:**
> "Software Developer with 3 years of experience in full-stack web development. Proficient in Python (Django/Flask) and JavaScript (React), with a track record of delivering clean, efficient code and improving application performance. Eager to contribute to a collaborative team building scalable software solutions."

### Experience
Transform your bullet points from tasks into achievements. Use the **Action Verb + What you did + Result/Impact** formula. Be specific.

**Current Bullet Point:**
> - Wrote code for web applications

**Rewritten Example:**
> - Engineered and launched 5 new user-facing features for a high-traffic e-commerce platform using Python and React, contributing to a 10% increase in user session duration.

**Current Bullet Point:**
> - Fixed bugs

**Rewritten Example:**
> - Resolved over 50 critical and major bugs in the production environment, improving application stability by 25% according to internal monitoring tools.

### Skills
A simple list is okay, but categorizing your skills makes it much easier for recruiters and hiring managers to scan. This shows you understand the different components of a tech stack.

**Rewritten Example:**
> **SKILLS**
> - **Programming Languages:** Python, JavaScript, SQL, HTML/CSS
> - **Frameworks & Libraries:** React, Django, Flask, Node.js *(add any you have used)*
> - **Databases:** PostgreSQL, MySQL, MongoDB *(be specific)*
> - **Tools & Platforms:** Git, Docker, Jenkins, AWS, Jira

### Education
This section is fine, but you can make it slightly more professional by being precise with the date.

**Rewritten Example:**
> **EDUCATION**
> **Bachelor of Science in Computer Science** - University of Milan
> *Graduated: October 2021*

## Missing Elements
- **GitHub/Portfolio Link:** This is non-negotiable for a developer. Your code is your proof. Add a clickable link to your GitHub, GitLab, or personal portfolio site right under your contact information.
- **Projects Section:** Especially with only 3 years of experience, a "Projects" section is crucial. Include 1-2 personal or academic projects with a brief description of the tech used and a link to the code/live demo. This shows passion and initiative beyond your paid work.
- **Specific Methodologies:** There is no mention of Agile, Scrum, Kanban, CI/CD, or testing frameworks (like Jest, PyTest). These are essential keywords that show you understand modern development workflows.

## ATS Compatibility
- **Rating: Medium**
- The CV uses standard headings (Summary, Experience), which is good for Applicant Tracking Systems. However, it lacks the keyword density that most systems are programmed to look for. A recruiter searching for "Agile," "REST API," or "CI/CD" would not find your CV.

- **Keyword Improvements:** Integrate these terms naturally into your experience bullet points:
    - Instead of "Worked with the team on projects," say "**Collaborated in an Agile/Scrum environment** to deliver features in two-week sprints."
    - Instead of "Learned about APIs," say "Assisted in the development and testing of **RESTful APIs** for a mobile application backend."
    - Mention specific tools like **Docker, Jenkins, Jira,** or cloud platforms like **AWS/GCP/Azure** if you have any experience with them.

## Targeted Review for a Specific Job

You can also ask the models to review your CV against a specific job posting.

In [7]:
def review_cv_for_job(cv_text, job_description, model_name, model_id):
    """Review a CV against a specific job posting."""
    job_prompt = f"""
You are an expert career coach. Compare this CV against the job description below.

Provide:
## Match Score: X/10

## Matching Skills & Experience
- List what aligns well

## Gaps to Address
- What's missing or weak for this specific role

## Tailoring Suggestions
- How to rewrite specific bullet points to better match this job
- Keywords to add from the job description

Be specific and reference exact parts of the CV and job description.
"""
    
    print(f"Asking {model_name}...")
    try:
        response = router_client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": job_prompt},
                {"role": "user", "content": f"CV:\n{cv_text}\n\nJOB DESCRIPTION:\n{job_description}"}
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error with {model_name}: {e}"

In [8]:
# Paste a job description here to get targeted feedback
job_description = """
We are looking for a Mid-Level Python Developer to join our AI team.

Requirements:
- 3+ years of Python experience
- Experience with REST APIs and microservices
- Familiarity with machine learning libraries (scikit-learn, PyTorch)
- Experience with cloud platforms (AWS or GCP)
- Strong communication skills
- CI/CD and Docker experience is a plus
"""

# Get review from one model (change the model if you prefer another)
model_name = "Claude Sonnet"
model_id = MODELS[model_name]
targeted_review = review_cv_for_job(my_cv, job_description, model_name, model_id)
display(Markdown(f"# {model_name} - Targeted Review\n{targeted_review}"))

Asking Claude Sonnet...


# Claude Sonnet - Targeted Review
## Match Score: 5/10

## Matching Skills & Experience
- **Python experience**: CV shows 3+ years of Python usage (current role since Jan 2022 + intern experience)
- **Team collaboration**: Mentions "worked with the team on projects"
- **API exposure**: Intern experience mentions "learned about APIs"
- **Technical foundation**: Computer Science degree and relevant programming languages
- **Communication potential**: B2 English level shows international communication capability

## Gaps to Address
- **No AI/ML experience**: Zero mention of machine learning, scikit-learn, or PyTorch
- **Missing cloud platforms**: No AWS or GCP experience mentioned
- **No microservices experience**: Current experience seems focused on monolithic web apps
- **Lack of CI/CD/Docker**: These valuable skills are completely absent
- **Vague technical descriptions**: Current bullets don't demonstrate depth of Python expertise

## Tailoring Suggestions

### Rewrite Experience Bullets:
**Current**: "Wrote code for web applications"
**Better**: "Developed Python-based web applications using REST APIs and integrated third-party services"

**Current**: "Fixed bugs"
**Better**: "Debugged and optimized Python applications, improving performance by implementing efficient algorithms"

**Current**: "Helped with development tasks"
**Better**: "Contributed to API development and testing, gaining hands-on experience with RESTful services"

### Keywords to Add:
- REST APIs / RESTful services
- Microservices architecture
- Machine learning (even if self-taught/learning)
- Cloud platforms
- API integration
- Data processing
- Algorithm optimization

### Summary Rewrite:
"Python developer with 3+ years of experience building web applications and REST APIs. Experienced in full-stack development with strong foundation in backend services. Eager to transition into AI development and expand expertise in machine learning technologies."

### Additional Recommendations:
1. Add any personal projects involving APIs or data processing
2. Mention any online courses in ML/AI you've taken
3. Include specific Python frameworks (Django, Flask, FastAPI)
4. Quantify achievements where possible (e.g., "improved application response time by 30%")

## Key Takeaways

1. **OpenRouter** provides a single API to access models from OpenAI, Anthropic, Google, and more
2. **Same prompt, different models** - each LLM has different strengths (creativity, structure, detail)
3. **Structured prompts** with a fixed output format make model comparison fair
4. **Benchmarking on real tasks** is more useful than synthetic benchmarks for choosing the right model
5. The OpenAI client library works with any OpenAI-compatible API by just changing `base_url`